# 03 — Features and the pipeline

Week 3. Everything here is knowable the moment the passenger gets in, every
learned step is fitted on **May only inside a pipeline**, and every addition
is judged on **June**.

The features that lost are reported too — further down, in their own table.
A feature list that only shows winners hides how the choices were made.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

from sklearn.tree import DecisionTreeRegressor

from src.data import rebuild, split
from src.evaluate import ladder, save_figure, score
from src.features import (
    CATEGORICAL,
    LEAKY,
    NUMERIC,
    build_features,
    build_preprocessor,
    feature_frame,
)
from src.models import (
    MedianDuration,
    TypicalSpeed,
    build_model,
    decision_tree,
    linear_regression,
    save,
    shallow_tree,
)


In [ ]:
df = build_features(rebuild())
parts = split(df)
train, val = parts["train"], parts["validation"]
y_train, y_val = train["duration_min"], val["duration_min"]
print({k: len(v) for k, v in parts.items()})
df[NUMERIC + CATEGORICAL].head()

## The pipeline cannot leak

Two guards, not one.

1. **An allow-list.** `src.features.feature_frame` selects only the columns
   named in `NUMERIC` and `CATEGORICAL`; everything else is invisible to the
   model, so adding a column to the dataframe cannot accidentally add it to
   the model.
2. **A deny-list that raises.** `assert_no_leakage` refuses anything in
   `LEAKY` — the drop-off time, the fare, the tip, the total, every fee,
   `payment_type`, `RatecodeID`, and the flags built from them.

In [ ]:
print("refused by the guard:")
print(", ".join(LEAKY))

try:
    feature_frame(train, numeric=NUMERIC + ["fare_amount"])
except ValueError as exc:
    print("\nblocked:", exc)

Every learned step — the median imputer, the scaler, the one-hot encoder —
sits **inside** the `ColumnTransformer`, so `.fit()` on May is the only place
anything is learned, and June and July are only ever transformed.

In [ ]:
pre = build_preprocessor()
pre.fit(feature_frame(train), y_train)
print(pre)
print(f"\n{len(pre.get_feature_names_out())} columns after the transform")

## One feature at a time, judged on June

Each row adds one group to the row above it. The estimator is held fixed (a
depth-8 tree) so what changes is the feature, not the model.

In [ ]:
CLOCK = ["pickup_hour", "pickup_dow", "pickup_minutes_into_day"]

steps = [
    ("1. distance only",  ["trip_distance"], []),
    ("2. + clock",        ["trip_distance"] + CLOCK, ["is_weekend"]),
    ("3. + boroughs",     ["trip_distance"] + CLOCK, ["is_weekend", "pickup_borough", "dropoff_borough"]),
    ("4. + airports",     ["trip_distance"] + CLOCK, ["is_weekend", "pickup_borough", "dropoff_borough",
                                                      "pickup_airport", "dropoff_airport"]),
    ("5. + passengers",   NUMERIC, CATEGORICAL),
]

rows = {}
for name, num, cat in steps:
    model = build_model(build_preprocessor(num, cat), DecisionTreeRegressor(max_depth=8, random_state=0))
    model.fit(train[num + cat], y_train)
    rows[name] = score(y_val, model.predict(val[num + cat]))

additions = ladder(rows).round(3)
additions

### The features that lost

An addition earns its place by moving June MAE. These did not, and they are
reported rather than quietly dropped.

In [ ]:
losers = additions.loc[additions["mae_vs_previous"] >= -0.01, ["model", "mae", "mae_vs_previous"]]
print("additions that did not pay for themselves (June MAE moved by < 0.01 min):")
print(losers.to_string(index=False) if len(losers) else "none")

# Tried and rejected: things that looked plausible and were not kept.
rejected = {}
base_num, base_cat = NUMERIC, CATEGORICAL
base = build_model(build_preprocessor(), DecisionTreeRegressor(max_depth=8, random_state=0))
base.fit(feature_frame(train), y_train)
rejected["baseline feature set"] = score(y_val, base.predict(feature_frame(val)))

# (a) the pickup zone itself, 260+ categories instead of the borough
zone_cat = [c for c in base_cat if c != "pickup_borough"] + ["pickup_zone"]
m = build_model(build_preprocessor(base_num, zone_cat), DecisionTreeRegressor(max_depth=8, random_state=0))
m.fit(train[base_num + zone_cat], y_train)
rejected["pickup_zone instead of borough"] = score(y_val, m.predict(val[base_num + zone_cat]))

# (b) distance squared, in case the relationship bends
train2 = train.assign(distance_sq=train["trip_distance"] ** 2)
val2 = val.assign(distance_sq=val["trip_distance"] ** 2)
num_sq = base_num + ["distance_sq"]
m = build_model(build_preprocessor(num_sq, base_cat), DecisionTreeRegressor(max_depth=8, random_state=0))
m.fit(train2[num_sq + base_cat], y_train)
rejected["+ distance squared"] = score(y_val, m.predict(val2[num_sq + base_cat]))

# (c) day of month, a feature with no reason behind it
train3 = train.assign(day_of_month=train["tpep_pickup_datetime"].dt.day)
val3 = val.assign(day_of_month=val["tpep_pickup_datetime"].dt.day)
num_dom = base_num + ["day_of_month"]
m = build_model(build_preprocessor(num_dom, base_cat), DecisionTreeRegressor(max_depth=8, random_state=0))
m.fit(train3[num_dom + base_cat], y_train)
rejected["+ day of month"] = score(y_val, m.predict(val3[num_dom + base_cat]))

ladder(rejected).round(3)

The clock is almost the whole gain. Adding the pickup hour, day of week and
minutes into the day cut June MAE by **0.58 minutes**; the boroughs added
0.07 after that, and nothing else moved the number at all. That matches what
`02` found: the city's daily rhythm is the strongest thing we can know at
pickup once the distance is accounted for.

**Two additions did not earn their place.** The airport flags moved June MAE
by 0.001 minutes and the passenger count by 0.000 — the latter is
unsurprising, since a quarter of the trips are flex fare and have no value
for it. We kept both, and that is a choice rather than a result: they cost
nothing to compute, and the airport flags make the airport slice in `04`
readable as a group the model has an explicit handle on. A stricter reading
of "an addition must pay for itself" would take them both back out of
`NUMERIC` and `CATEGORICAL` in `src/features.py`.

**Three ideas were tried and rejected**, and they are in the table above
rather than deleted from history. The pickup zone instead of the borough cost
0.011 minutes for 260-odd extra categories — more detail, worse predictions,
which is what overfitting a split looks like. Distance squared changed
nothing, as expected: a tree splits on order, so a monotone transform of a
feature it already has tells it nothing new. The day of the month made things
worse by 0.027 — a feature with no reason behind it, included precisely to
check that our procedure can reject something, and it did.

## The models

Linear regression and a single decision tree — the two families the project
allows — against the ladder from `01`. The tree's depth is chosen on June,
which is what June is for.

In [ ]:
X_train, X_val = feature_frame(train), feature_frame(val)
results = {}

results["1. median of May"] = score(y_val, MedianDuration().fit(X_train, y_train).predict(X_val))

rung2 = TypicalSpeed().fit(train[["trip_distance"]], y_train)
print(f"typical speed fitted on May: {rung2.speed_mph_:.2f} mph")
results["2. distance / typical speed"] = score(y_val, rung2.predict(val[["trip_distance"]]))

results["3. tree, depth 2"] = score(
    y_val, build_model(build_preprocessor(), shallow_tree(2)).fit(X_train, y_train).predict(X_val))

linreg = build_model(build_preprocessor(), linear_regression()).fit(X_train, y_train)
results["4. linear regression"] = score(y_val, linreg.predict(X_val))

for depth in (4, 6, 8, 10, 12, 14, 16):
    tree = build_model(build_preprocessor(), decision_tree(depth)).fit(X_train, y_train)
    results[f"5. tree, depth {depth}"] = score(y_val, tree.predict(X_val))

ladder(results).round(3)

In [ ]:
depths = {int(k.split()[-1]): v["mae"] for k, v in results.items() if k.startswith("5.")}
best_depth = min(depths, key=depths.get)
print(f"best depth on June: {best_depth} (MAE {depths[best_depth]:.3f})")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(list(depths), list(depths.values()), marker="o")
ax.axvline(best_depth, ls="--", lw=1)
ax.set(xlabel="tree depth", ylabel="June MAE (min)", title="Depth chosen on June")
save_figure(fig, "tree-depth-on-june")
plt.show()

In [ ]:
chosen = build_model(build_preprocessor(), decision_tree(best_depth)).fit(X_train, y_train)
print(save(chosen, "chosen"))
print(save(linreg, "linear-regression"))
print(f"\nchosen: decision tree, depth {best_depth}, fitted on May only")

### What `03` settles

The chosen model is a decision tree of **depth 12**, scoring **4.584 MAE**
and 6.967 RMSE on June. It beats the depth-2 tree from the ladder by 1.43
minutes and the linear regression by 1.05, so the extra depth is doing real
work rather than memorising May.

Against the threshold in `PROBLEM.md` — beat rung 2 by at least a minute of
MAE, with a smaller buffer — the margin is **5.09 minutes** (4.584 against
9.672). That clears the bar by a wide margin, and it is worth saying plainly
why the bar was easy: rung 2 is the *worst* model in the ladder on MAE, worse
even than predicting the median of May for every trip, because one typical
speed badly overestimates long trips. Measured against rung 1 instead, the
margin is 4.28 minutes, which is the number we would defend.

Depth was chosen on June and nowhere else. Depths 14 and 16 scored 4.664 and
4.853 — worse — so 12 is a real optimum rather than the end of the range we
happened to try.